# Assignment: Exploring Latent Spaces with VAEs and GANs

**Platform:** NRP Nautilus JupyterHub (request **1 GPU**)
**Dataset:** Fashion-MNIST (10 clothing classes, 28×28 grayscale)
**Total:** 100 points (+10 bonus) · **Estimated GPU time:** ~12 min total

### Instructions
* Fill in every cell marked `# YOUR CODE HERE` (delete the `raise NotImplementedError`) and every **Your answer** cell.
* Run the **sanity-check** cells after each TODO; they must pass before moving on.
* Do **not** change function names or signatures.
* Before submitting: `Kernel → Restart Kernel and Run All Cells`, make sure everything runs top-to-bottom, then
  download the notebook (`File → Download`) **and** an HTML export (`File → Save and Export Notebook As → HTML`).

| Part | Topic | Points |
|---|---|---|
| 1 | Implement & train a VAE | 25 |
| 2 | Explore the VAE latent space | 35 |
| 3 | Implement & train a GAN; mode coverage | 25 |
| 4 | Written reflection | 15 |
| Bonus | Convolutional GAN (DCGAN) | +10 |

Fashion-MNIST labels: `0 T-shirt · 1 Trouser · 2 Pullover · 3 Dress · 4 Coat · 5 Sandal · 6 Shirt · 7 Sneaker · 8 Bag · 9 Ankle boot`

## Part 0 — Setup (given)

In [ ]:
import os, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

DATA_DIR = os.path.expanduser("~/data")
CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

train_ds = torchvision.datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=T.ToTensor())
test_ds  = torchvision.datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=T.ToTensor())
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0, drop_last=True)
test_loader  = DataLoader(test_ds, batch_size=512, shuffle=False, num_workers=0)

def show_grid(images, nrow=10, title=None, figsize=(8, 8)):
    grid = torchvision.utils.make_grid(images.detach().cpu(), nrow=nrow, padding=2, pad_value=1.0)
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

show_grid(torch.stack([train_ds[i][0] for i in range(30)]), title="Fashion-MNIST samples", figsize=(8, 3))

---
## Part 1 — Implement and train a VAE (25 pts)

### TODO 1.1 (5 pts) — Reparameterization trick
Implement `reparameterize`: given `mu` and `logvar` (= $\log\sigma^2$), return $\mathbf{z} = \boldsymbol\mu + \boldsymbol\sigma\odot\boldsymbol\epsilon$ with $\boldsymbol\epsilon\sim\mathcal{N}(0, I)$.

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2, hidden=512):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, hidden), nn.ReLU(),
            nn.Linear(hidden, 256), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, hidden), nn.ReLU(),
            nn.Linear(hidden, 784),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        # YOUR CODE HERE
        raise NotImplementedError

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)  # logits

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

### TODO 1.2 (10 pts) — VAE loss
Return `(total, recon, kl)` where, **averaged over the batch** (sum over pixels / latent dims, then divide by batch size):
* `recon` = binary cross-entropy between the decoder **logits** and `x` (hint: `F.binary_cross_entropy_with_logits`)
* `kl` = $-\tfrac12\sum_j \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$
* `total` = `recon + beta * kl`

In [ ]:
def vae_loss(logits, x, mu, logvar, beta=1.0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# ---- Sanity checks for 1.1 and 1.2 (do not edit) ----
_m = VAE(latent_dim=4)
_mu = torch.randn(1000, 4)
_z = _m.reparameterize(_mu, torch.full((1000, 4), -30.0))       # sigma ~ 0  ->  z ~ mu
assert torch.allclose(_z, _mu, atol=1e-4), "With tiny variance, z should equal mu"
_z = _m.reparameterize(torch.zeros(20000, 4), torch.zeros(20000, 4))  # N(0, 1)
assert abs(_z.mean().item()) < 0.05 and abs(_z.std().item() - 1) < 0.05, "z should be ~N(0,1) when mu=0, logvar=0"

_x = torch.rand(8, 1, 28, 28)
_tot, _rec, _kl = vae_loss(torch.zeros(8, 1, 28, 28), _x, torch.zeros(8, 4), torch.zeros(8, 4))
assert abs(_kl.item()) < 1e-6, "KL(N(0,1) || N(0,1)) must be 0"
assert abs(_rec.item() - 784 * np.log(2)) < 1e-3, "recon should be summed over pixels and averaged over the batch"
_tot, _rec, _kl = vae_loss(torch.zeros(8, 1, 28, 28), _x, torch.ones(8, 4), torch.zeros(8, 4), beta=2.0)
assert abs(_kl.item() - 2.0) < 1e-5, "KL for mu=1, sigma=1 in 4 dims should be 2.0"
assert abs(_tot.item() - (_rec.item() + 2.0 * _kl.item())) < 1e-4, "total = recon + beta * kl"
print("✅ All Part 1 sanity checks passed")

### 1.3 (given) Training function
Read through it — you will call it several times.

In [ ]:
def train_vae(latent_dim, epochs=15, beta=1.0, verbose=True):
    model = VAE(latent_dim).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(1, epochs + 1):
        model.train()
        tot = rec = kld = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            logits, mu, logvar = model(x)
            loss, r, k = vae_loss(logits, x, mu, logvar, beta)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item(); rec += r.item(); kld += k.item()
        n = len(train_loader)
        if verbose:
            print(f"[d={latent_dim}, beta={beta}] epoch {epoch:2d} | loss {tot/n:7.2f} | recon {rec/n:7.2f} | KL {kld/n:6.2f}")
    model.eval()
    return model, rec / n, kld / n

@torch.no_grad()
def encode_dataset(model, loader=test_loader):
    model.eval()
    zs, ys = [], []
    for x, y in loader:
        mu, _ = model.encode(x.to(device))
        zs.append(mu.cpu()); ys.append(y)
    return torch.cat(zs), torch.cat(ys)

In [ ]:
VAE_EPOCHS = 15
vae2, rec2, kl2 = train_vae(latent_dim=2, epochs=VAE_EPOCHS)
vae16, rec16, kl16 = train_vae(latent_dim=16, epochs=VAE_EPOCHS)

In [ ]:
with torch.no_grad():
    x = torch.stack([test_ds[i][0] for i in range(10)]).to(device)
    r2 = torch.sigmoid(vae2.decode(vae2.encode(x)[0]))
    r16 = torch.sigmoid(vae16.decode(vae16.encode(x)[0]))
show_grid(torch.cat([x, r2, r16]), nrow=10, title="Rows: original | d=2 reconstruction | d=16 reconstruction", figsize=(10, 3.5))
print(f"Final train recon loss  d=2: {rec2:.1f}   d=16: {rec16:.1f}")
print(f"Final train KL          d=2: {kl2:.2f}   d=16: {kl16:.2f}")

**Question 1.4 (5 pts).** Compare the $d=2$ and $d=16$ models. Which reconstructs better and why? Why is the KL term larger for $d=16$?
What do we *give up* by using a larger latent space?

**Your answer:**

_(double-click to edit)_

### TODO 1.5 (5 pts) — Generate new clothing
Sample **50** latent vectors from the prior $\mathcal{N}(0, I)$, decode them with `vae16`, convert logits to pixel values, and store them in `vae16_samples` with shape `(50, 1, 28, 28)`.

In [ ]:
with torch.no_grad():
    # YOUR CODE HERE
    raise NotImplementedError

assert vae16_samples.shape == (50, 1, 28, 28)
assert 0 <= vae16_samples.min() and vae16_samples.max() <= 1, "Did you apply sigmoid to the logits?"
show_grid(vae16_samples, title="VAE (d=16) samples from the prior", figsize=(8, 4.5))

---
## Part 2 — Exploring the VAE latent space (35 pts)

### 2.1 (given) The 2-D latent space

In [ ]:
Z2, Y2 = encode_dataset(vae2)
plt.figure(figsize=(8, 6.5))
for c in range(10):
    m = Y2 == c
    plt.scatter(Z2[m, 0], Z2[m, 1], s=3, alpha=0.6, label=CLASSES[c])
plt.legend(markerscale=4, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xlabel("$z_1$"); plt.ylabel("$z_2$"); plt.title("VAE latent space, d=2 (Fashion-MNIST test set)")
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

**Question 2.1 (4 pts).** Which classes form clean clusters and which overlap? Give a visual reason why the overlapping classes are confused.
Remember: the VAE was never given labels.

**Your answer:**

_(double-click to edit)_

### TODO 2.2 (6 pts) — Visualise the 16-D latent space with PCA
We can't plot 16 dimensions directly. Implement `pca_2d(Z)`: **center** `Z` (subtract the column mean), compute its top-2 principal directions
(e.g. with `torch.linalg.svd(..., full_matrices=False)`), and return the `(N, 2)` projection.

In [ ]:
def pca_2d(Z):
    # YOUR CODE HERE
    raise NotImplementedError

# sanity check
_Z = torch.randn(500, 16) * torch.tensor([5.0, 3.0] + [0.1] * 14)
_P = pca_2d(_Z)
assert _P.shape == (500, 2)
assert abs(_P.mean().item()) < 1e-4, "projection should be centered"
assert _P[:, 0].var() > _P[:, 1].var() > 1.0, "first component should explain the most variance"
print("✅ pca_2d check passed")

Z16, Y16 = encode_dataset(vae16)
P16 = pca_2d(Z16)
plt.figure(figsize=(8, 6.5))
for c in range(10):
    m = Y16 == c
    plt.scatter(P16[m, 0], P16[m, 1], s=3, alpha=0.6, label=CLASSES[c])
plt.legend(markerscale=4, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.title("VAE latent space, d=16, projected with PCA"); plt.tight_layout(); plt.show()

### TODO 2.3 (8 pts) — Latent interpolation
Implement `interpolate(model, xa, xb, steps)`:
1. Encode `xa` and `xb` (single images of shape `(1, 28, 28)`) to their **means** $\mathbf{z}_a, \mathbf{z}_b$.
2. Build `steps` evenly spaced points $\mathbf{z}(t) = (1-t)\mathbf{z}_a + t\,\mathbf{z}_b$ for $t\in[0,1]$.
3. Decode and return pixel images of shape `(steps, 1, 28, 28)`.

In [ ]:
@torch.no_grad()
def interpolate(model, xa, xb, steps=10):
    model.eval()
    # YOUR CODE HERE
    raise NotImplementedError

def first_of(label):
    return next(test_ds[i][0] for i in range(len(test_ds)) if test_ds[i][1] == label)

_out = interpolate(vae16, first_of(7), first_of(9), steps=8)
assert _out.shape == (8, 1, 28, 28)
with torch.no_grad():
    _end = torch.sigmoid(vae16.decode(vae16.encode(first_of(9).unsqueeze(0).to(device))[0]))
assert torch.allclose(_out[-1:], _end, atol=1e-5), "last frame should be the reconstruction of xb"

pairs = [(7, 9), (1, 3), (5, 7), (0, 8)]   # sneaker→boot, trouser→dress, sandal→sneaker, T-shirt→bag
rows = [interpolate(vae16, first_of(a), first_of(b)) for a, b in pairs]
show_grid(torch.cat(rows), title="; ".join(f"{CLASSES[a]}→{CLASSES[b]}" for a, b in pairs), figsize=(10, 4.5))

**Question 2.3 (4 pts).** Are the intermediate images realistic clothing items? Which interpolation looks most natural and which looks least
natural? Relate your answer to the class layout you saw in 2.1/2.2.

**Your answer:**

_(double-click to edit)_

### TODO 2.4 (6 pts) — Class centroids and latent arithmetic
Implement `class_centroids(Z, Y)` returning a `(10, d)` tensor whose row `c` is the mean latent vector of all test images with label `c`.
We then decode the centroids ("the average sneaker") and try **latent arithmetic**: add the direction
$\bar{\mathbf{z}}_\text{ankle boot} - \bar{\mathbf{z}}_\text{sneaker}$ to individual sneakers.

In [ ]:
def class_centroids(Z, Y, num_classes=10):
    # YOUR CODE HERE
    raise NotImplementedError

cent = class_centroids(Z16, Y16)
assert cent.shape == (10, Z16.shape[1])
assert torch.allclose(cent[3], Z16[Y16 == 3].mean(0)), "row c must be the mean of class c"
print("✅ class_centroids check passed")

with torch.no_grad():
    show_grid(torch.sigmoid(vae16.decode(cent.to(device))), nrow=10, title="Decoded class centroids", figsize=(10, 1.6))

    direction = (cent[9] - cent[7]).to(device)                 # sneaker -> ankle boot
    sneakers = Z16[Y16 == 7][:8].to(device)
    alphas = torch.linspace(0, 1.5, 7, device=device)
    rows = [torch.sigmoid(vae16.decode(z + alphas.unsqueeze(1) * direction)) for z in sneakers]
show_grid(torch.cat(rows), nrow=7, title="sneaker + α·(boot − sneaker),  α = 0 … 1.5", figsize=(6, 7))

### 2.5 (given) The effect of β
Train three 16-D VAEs with different $\beta$ (5 epochs each, ~1 min on GPU) and compare.

In [ ]:
beta_results = {}
for beta in [0.1, 1.0, 4.0]:
    model, rec, kl = train_vae(16, epochs=5, beta=beta, verbose=False)
    beta_results[beta] = (model, rec, kl)
    print(f"beta={beta:<4} | recon {rec:7.2f} | KL {kl:6.2f}")

with torch.no_grad():
    z = torch.randn(10, 16, device=device)
    x = torch.stack([test_ds[i][0] for i in range(10)]).to(device)
    rows = []
    for beta, (m, _, _) in beta_results.items():
        rows += [torch.sigmoid(m.decode(m.encode(x)[0])), torch.sigmoid(m.decode(z))]
show_grid(torch.cat(rows), nrow=10,
          title="Pairs of rows per β (0.1, 1, 4): reconstructions, then prior samples", figsize=(10, 6.5))

**Question 2.5 (7 pts).** Describe the trade-off you observe as $\beta$ increases, both in the numbers and in the images.
Why do prior samples from the $\beta=0.1$ model tend to look worse than its reconstructions? Connect your answer to the
"continuous" and "complete" properties of a latent space from the lecture.

**Your answer:**

_(double-click to edit)_

---
## Part 3 — GANs and mode coverage (25 pts)

The networks and optimisers are given. Implement the two halves of one training step.

In [ ]:
Z_DIM = 64
to_gan = lambda x: x * 2 - 1
from_gan = lambda x: (x + 1) / 2

class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.BatchNorm1d(512), nn.LeakyReLU(0.2),
            nn.Linear(512, 1024), nn.BatchNorm1d(1024), nn.LeakyReLU(0.2),
            nn.Linear(1024, 784), nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x)

bce = nn.BCEWithLogitsLoss()

### TODO 3.1 (5 pts) — Discriminator step
Compute `d_loss = BCE(D(real), 1) + BCE(D(fake), 0)`, then zero the gradients of `opt_D`, backpropagate, and step.
**Important:** the generator must not receive gradients here — detach `fake`.

### TODO 3.2 (5 pts) — Generator step (non-saturating loss)
Compute `g_loss = BCE(D(fake), 1)`, then zero the gradients of `opt_G`, backpropagate, and step. Return the loss values as Python floats.

In [ ]:
def discriminator_step(G, D, opt_D, real):
    b = real.size(0)
    fake = G(torch.randn(b, Z_DIM, device=real.device))
    # YOUR CODE HERE
    raise NotImplementedError
    return d_loss.item(), fake

def generator_step(G, D, opt_G, fake):
    b = fake.size(0)
    # YOUR CODE HERE
    raise NotImplementedError
    return g_loss.item()

In [ ]:
# ---- Sanity check for 3.1 / 3.2 (do not edit) ----
torch.manual_seed(0)
_G, _D = Generator().to(device), Discriminator().to(device)
_oG, _oD = torch.optim.Adam(_G.parameters(), 2e-4), torch.optim.Adam(_D.parameters(), 2e-4)
_g0 = [p.clone() for p in _G.parameters()]
_d0 = [p.clone() for p in _D.parameters()]
_real = to_gan(next(iter(train_loader))[0].to(device))
_dl, _fake = discriminator_step(_G, _D, _oD, _real)
assert all(torch.equal(a, b) for a, b in zip(_g0, _G.parameters())), "D step must not update G (did you detach?)"
assert any(not torch.equal(a, b) for a, b in zip(_d0, _D.parameters())), "D step must update D"
_d1 = [p.clone() for p in _D.parameters()]
_gl = generator_step(_G, _D, _oG, _fake)
assert any(not torch.equal(a, b) for a, b in zip(_g0, _G.parameters())), "G step must update G"
assert all(torch.equal(a, b) for a, b in zip(_d1, _D.parameters())), "G step must not update D"
assert isinstance(_dl, float) and isinstance(_gl, float)
print(f"✅ GAN step checks passed (d_loss={_dl:.3f}, g_loss={_gl:.3f})")

In [ ]:
GAN_EPOCHS = 30   # ~4 min on a GPU
G, D = Generator().to(device), Discriminator().to(device)
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
fixed_z = torch.randn(50, Z_DIM, device=device)
gan_hist = []

for epoch in range(1, GAN_EPOCHS + 1):
    G.train(); D.train()
    ds = gs = 0.0
    for x, _ in train_loader:
        d_l, fake = discriminator_step(G, D, opt_D, to_gan(x.to(device)))
        gs += generator_step(G, D, opt_G, fake)
        ds += d_l
    gan_hist.append((ds / len(train_loader), gs / len(train_loader)))
    print(f"epoch {epoch:2d} | D {gan_hist[-1][0]:.3f} | G {gan_hist[-1][1]:.3f}")
    if epoch % 10 == 0 or epoch == GAN_EPOCHS:
        G.eval()
        with torch.no_grad():
            show_grid(from_gan(G(fixed_z)), title=f"GAN samples, epoch {epoch}", figsize=(8, 4.5))

### 3.3 (given) Measuring mode coverage
A GAN that only produces T-shirts and trousers could still look great sample-by-sample. To detect **mode collapse** we train a quick
classifier on real data, then use it to label 5,000 generated images from each model.
A model that covers all modes should produce a roughly uniform histogram (~10% per class).

In [ ]:
clf = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10),
).to(device)
opt_c = torch.optim.Adam(clf.parameters(), 1e-3)
for _ in range(3):
    clf.train()
    for x, y in train_loader:
        loss = F.cross_entropy(clf(x.to(device)), y.to(device))
        opt_c.zero_grad(); loss.backward(); opt_c.step()
clf.eval()
with torch.no_grad():
    acc = np.mean([(clf(x.to(device)).argmax(1).cpu() == y).float().mean().item() for x, y in test_loader])
print(f"classifier test accuracy: {acc:.3f}")

@torch.no_grad()
def class_histogram(sample_fn, n=5000):
    preds = torch.cat([clf(sample_fn(500)).argmax(1).cpu() for _ in range(n // 500)])
    return torch.bincount(preds, minlength=10).float() / len(preds)

G.eval()
hist_gan = class_histogram(lambda k: from_gan(G(torch.randn(k, Z_DIM, device=device))))
hist_vae = class_histogram(lambda k: torch.sigmoid(vae16.decode(torch.randn(k, 16, device=device))))

xs = np.arange(10)
plt.figure(figsize=(11, 3.8))
plt.bar(xs - 0.2, hist_vae, 0.4, label="VAE (d=16)")
plt.bar(xs + 0.2, hist_gan, 0.4, label="GAN")
plt.axhline(0.1, color="k", ls="--", lw=1, label="uniform")
plt.xticks(xs, CLASSES, rotation=30); plt.ylabel("fraction of samples"); plt.legend(); plt.tight_layout(); plt.show()

entropy = lambda p: -(p[p > 0] * p[p > 0].log()).sum().item() / np.log(10)
print(f"normalised class entropy (1.0 = perfectly uniform):  VAE {entropy(hist_vae):.3f} | GAN {entropy(hist_gan):.3f}")

### TODO 3.4 (5 pts) — GAN interpolation
Use the provided `slerp` to interpolate between **4 pairs** of random noise vectors (10 steps each) and display them with `show_grid`.

In [ ]:
def slerp(a, b, t):
    a_n, b_n = a / a.norm(dim=-1, keepdim=True), b / b.norm(dim=-1, keepdim=True)
    omega = torch.acos((a_n * b_n).sum(-1, keepdim=True).clamp(-1, 1))
    so = torch.sin(omega)
    return (torch.sin((1 - t) * omega) / so) * a + (torch.sin(t * omega) / so) * b

G.eval()
with torch.no_grad():
    # YOUR CODE HERE
    raise NotImplementedError

**Question 3.5 (5 pts).** Compare the GAN and VAE (d=16) samples: which are sharper, and which have better class coverage according to the
histogram? Is there evidence of (partial) mode collapse? Why can't we draw the GAN's latent space the way we did in 2.1?

**Your answer:**

_(double-click to edit)_

---
## Part 4 — Written reflection (15 pts)

Answer each in 3–6 sentences.

**4.1 (5 pts).** In your own words, define *latent space*. Using evidence from **your** plots, explain what "continuous" and "complete" mean
and which model's latent space better satisfies each property.

**4.2 (5 pts).** You are building a system that flags defective products from images by detecting inputs that are "unusual".
Would you build it on a VAE or a GAN? Justify using what each model can and cannot do (hint: which one has an encoder and a likelihood bound?).

**4.3 (5 pts).** Modern text-to-image systems (e.g. Stable Diffusion) run diffusion **inside the latent space of an autoencoder** rather than in
pixel space. Based on this assignment, give two reasons why working in a latent space is attractive.

**Your answer:**

_(double-click to edit)_

---
## Bonus (+10 pts) — DCGAN
Replace the MLP generator/discriminator with **convolutional** ones (e.g. `ConvTranspose2d` in G: 64→(128,7,7)→(64,14,14)→(1,28,28);
strided `Conv2d` in D) and retrain using your `discriminator_step` / `generator_step`. Show samples, re-run the class histogram from 3.3,
and write 2–3 sentences comparing against the MLP GAN.

In [ ]:
# Bonus: your DCGAN code here (optional)